In [20]:
# ==============================================================================
# PROYECTO PyEconGDL
# DE FUENTES CRUDAS HETEROGÉNEAS A UN DATASET CONSOLIDADO EN PARQUET
#
# RUTA CONCEPTUAL:
#
# FUENTES CRUDAS (CSV / XLSX)
#   -> INSPECCIÓN DE ESTRUCTURA Y CODIFICACIÓN (UTF-8 vs LATIN-1)
#   -> SELECCIÓN DE SERIES COMPARABLES (GRANULARIDAD Y UNIDADES)
#   -> ESTANDARIZACIÓN TEMPORAL (CICLO: AÑO-MES)
#   -> PERSISTENCIA INDIVIDUAL EN PARQUET (RESPETANDO COBERTURA PROPIA)
#   -> INTEGRACIÓN MEDIANTE INNER JOIN (2011-01 HASTA COBERTURA COMÚN)
#   -> VALIDACIÓN DE CALIDAD Y PRODUCTO ANALÍTICO FINAL
#
# PRINCIPIOS DEL CURSO:
# - DATO -> SIGNIFICADO -> OPERACIÓN -> INTERPRETACIÓN
# - CÓDIGO CORRECTO != ANÁLISIS CORRECTO
# - OBSERVAR ANTES DE TRANSFORMAR
# ==============================================================================

In [21]:
# ==============================================================================
# 0. PREPARACIÓN: EJECUTAR DESDE LA RAÍZ DEL PROYECTO (PyEconGDL)
# ==============================================================================
#
# OBJETIVO:
# Garantizar que este notebook funcione de manera reproducible en cualquier
# computadora que conserve la estructura de carpetas del proyecto PyEconGDL:
#
#   PyEconGDL/
#   ├── data/
#   │   ├── raw/         <- Archivos originales (CSV y Excel)
#   │   └── processed/   <- Parquets individuales y consolidado final
#   └── notebooks/       <- Este Jupyter Notebook
#
# CRITERIO:
# Nunca fijar rutas absolutas personales (/Users/...). Detectar dinámicamente
# la raíz del proyecto permite compartir el repositorio sin romper rutas.
# ------------------------------------------------------------------------------
from pathlib import Path
import os
import re
import time
import numpy as np
import pandas as pd
from IPython.display import display

pd.options.display.float_format = "{:,.4f}".format

# Si Jupyter abrió dentro de notebooks/, retrocedemos un nivel hacia PyEconGDL
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Directorio de trabajo (raíz):", PROJECT_ROOT)
print("Carpeta de entrada (raw):    ", RAW_DIR)
print("Carpeta de salida (processed):", PROCESSED_DIR)

Directorio de trabajo (raíz): C:\Users\micha\Documents\VS Code\PyEconGDL
Carpeta de entrada (raw):     C:\Users\micha\Documents\VS Code\PyEconGDL\data\raw
Carpeta de salida (processed): C:\Users\micha\Documents\VS Code\PyEconGDL\data\processed


In [22]:
# ==============================================================================
# 1. CATÁLOGO DE FUENTES CRUDAS Y VERIFICACIÓN DE EXISTENCIA
# ==============================================================================
#
# Antes de leer cualquier archivo, verificamos que los 5 insumos estén en
# data/raw/ con sus nombres oficiales:
#   1. conjunto_de_datos_inpc_mensual.csv
#   2. Consulta_20261031-181439737.csv
#   3. deuda_publica.csv
#   4. IGAE_2.csv.xlsx
#   5. ingreso_gasto_finan.csv (archivo completo ~250 MB de SHCP)
# ------------------------------------------------------------------------------
RUTA_INPC = RAW_DIR / "conjunto_de_datos_inpc_mensual.csv"
RUTA_TIIE = RAW_DIR / "Consulta_20261031-181439737.csv"
RUTA_DEUDA = RAW_DIR / "deuda_publica.csv"
RUTA_IGAE = RAW_DIR / "IGAE_2.csv.xlsx"
RUTA_INGRESO_GASTO = RAW_DIR / "ingreso_gasto_finan.csv"

fuentes_esperadas = {
    "1. INPC (INEGI)": RUTA_INPC,
    "2. TIIE Fondeo (Banxico)": RUTA_TIIE,
    "3. Deuda Pública (SHCP)": RUTA_DEUDA,
    "4. IGAE (INEGI)": RUTA_IGAE,
    "5. Ingreso y Gasto (SHCP)": RUTA_INGRESO_GASTO,
}

for etiqueta, ruta in fuentes_esperadas.items():
    existe = ruta.exists()
    tam_mb = round(ruta.stat().st_size / (1024 ** 2), 2) if existe else 0.0
    print(f"{etiqueta:<28} | Existe: {str(existe):<5} | {tam_mb:>8} MB | {ruta.name}")
    assert existe, f"Falta el archivo requerido en data/raw/: {ruta.name}"

1. INPC (INEGI)              | Existe: True  |     0.39 MB | conjunto_de_datos_inpc_mensual.csv
2. TIIE Fondeo (Banxico)     | Existe: True  |     0.08 MB | Consulta_20261031-181439737.csv
3. Deuda Pública (SHCP)      | Existe: True  |    23.44 MB | deuda_publica.csv
4. IGAE (INEGI)              | Existe: True  |     0.62 MB | IGAE_2.csv.xlsx
5. Ingreso y Gasto (SHCP)    | Existe: True  |   248.02 MB | ingreso_gasto_finan.csv


In [23]:
# ==============================================================================
# 2. FUENTE 1 — INPC MENSUAL (INEGI)
# ==============================================================================
#
# OBJETIVO:
# Extraer la serie general del Índice Nacional de Precios al Consumidor (INPC)
# en formato mensual, conservando su cobertura histórica completa.
#
# CONTRATO DE SALIDA INDIVIDUAL:
# - Archivo: data/processed/inpc_mensual.parquet
# - Columnas: CICLO (YYYY-MM), VALOR_INPC (float64, índice base 2ª quincena jul 2018=100)
# - Orden: ascendente por CICLO, sin duplicados.
# ------------------------------------------------------------------------------
df_inpc_raw = pd.read_csv(RUTA_INPC, encoding="utf-8")
print("Registros crudos INPC:", len(df_inpc_raw))
print("Conceptos disponibles:", df_inpc_raw["CONCEPTO"].nunique())

# Filtramos exclusivamente el índice general (evitando subíndices subyacente/no subyacente)
df_inpc = df_inpc_raw[
    df_inpc_raw["CONCEPTO"].str.endswith("Precios al Consumidor (INPC)", na=False)
].copy()

df_inpc["FECHA_DT"] = pd.to_datetime(df_inpc["FECHA"])
df_inpc["CICLO"] = df_inpc["FECHA_DT"].dt.strftime("%Y-%m")
df_inpc["VALOR_INPC"] = pd.to_numeric(df_inpc["VALOR"], errors="coerce")

inpc_parquet = (
    df_inpc[["CICLO", "VALOR_INPC"]]
    .dropna(subset=["CICLO", "VALOR_INPC"])
    .sort_values("CICLO", ascending=True)
    .drop_duplicates(subset=["CICLO"], keep="last")
    .reset_index(drop=True)
)

assert inpc_parquet["CICLO"].duplicated().sum() == 0
assert (inpc_parquet["VALOR_INPC"] > 0).all()

SALIDA_INPC = PROCESSED_DIR / "inpc_mensual.parquet"
inpc_parquet.to_parquet(SALIDA_INPC, index=False)

print(f"INPC limpio -> Filas: {len(inpc_parquet)} | Cobertura: {inpc_parquet['CICLO'].min()} a {inpc_parquet['CICLO'].max()}")
display(inpc_parquet.head(5))
display(inpc_parquet.tail(3))

Registros crudos INPC: 1988
Conceptos disponibles: 7
INPC limpio -> Filas: 284 | Cobertura: 2003-01 a 2026-08


,CICLO,VALOR_INPC
0,2003-01,53.5254
1,2003-02,53.6741
2,2003-03,54.0129
3,2003-04,54.1051
4,2003-05,53.9306


,CICLO,VALOR_INPC
281,2026-06,145.1310
282,2026-07,145.1690
283,2026-08,145.4620


In [24]:
# ==============================================================================
# 3. FUENTE 2 — TIIE DE FONDEO A 1 DÍA (BANXICO)
# ==============================================================================
#
# OBJETIVO:
# Leer la serie diaria SF331451 (TIIE de fondeo a un día hábil bancario),
# limpiar metadatos y valores 'N/E', filtrar desde 2011 y agregar a frecuencia
# mensual mediante el promedio aritmético del mes.
#
# CONTRATO DE SALIDA INDIVIDUAL:
# - Archivo: data/processed/tiie_mensual.parquet
# - Columnas: CICLO (YYYY-MM), TIIE (float64, tasa porcentual anualizada)
# - Orden: ascendente por CICLO, sin columna MES_NUM redundante.
# ------------------------------------------------------------------------------
df_tiie_raw = pd.read_csv(
    RUTA_TIIE,
    skiprows=18,
    encoding="latin-1",
    na_values=["N/E", "n/e", ""]
)

# Renombrar columnas (SF331451 es la clave de TIIE de Fondeo a 1 día)
df_tiie_raw.columns = ["FECHA", "TIIE_FONDEO"]

# Convertir tipos de datos y eliminar días inhábiles / nulos
df_tiie_raw["FECHA"] = pd.to_datetime(df_tiie_raw["FECHA"], format="%d/%m/%Y")
df_tiie_raw["TIIE_FONDEO"] = pd.to_numeric(df_tiie_raw["TIIE_FONDEO"], errors="coerce")
df_tiie_limpio = (
    df_tiie_raw
    .dropna(subset=["FECHA", "TIIE_FONDEO"])
    .sort_values("FECHA", ascending=True)
)

# Filtrar desde 2011 en adelante y construir CICLO (YYYY-MM)
df_tiie_2011 = df_tiie_limpio[df_tiie_limpio["FECHA"].dt.year >= 2011].copy()
df_tiie_2011["CICLO"] = df_tiie_2011["FECHA"].dt.strftime("%Y-%m")

# Agrupar por CICLO para obtener una observación mensual (promedio del mes)
tiie_parquet = (
    df_tiie_2011
    .groupby("CICLO", as_index=False)
    .agg(TIIE=("TIIE_FONDEO", "mean"))
    .sort_values("CICLO", ascending=True)
    .reset_index(drop=True)
)

assert tiie_parquet["CICLO"].duplicated().sum() == 0

SALIDA_TIIE = PROCESSED_DIR / "tiie_mensual.parquet"
tiie_parquet.to_parquet(SALIDA_TIIE, index=False)

print(f"TIIE mensual -> Filas: {len(tiie_parquet)} | Cobertura: {tiie_parquet['CICLO'].min()} a {tiie_parquet['CICLO'].max()}")
display(tiie_parquet.head(5))
display(tiie_parquet.tail(3))

TIIE mensual -> Filas: 190 | Cobertura: 2011-01 a 2026-10


,CICLO,TIIE
0,2011-01,4.5495
1,2011-02,4.5184
2,2011-03,4.4886
3,2011-04,4.5047
4,2011-05,4.4664


,CICLO,TIIE
187,2026-08,6.5043
188,2026-09,6.5162
189,2026-10,6.5191


In [25]:
# ==============================================================================
# 4. FUENTE 3 — DEUDA BRUTA DEL SECTOR PÚBLICO FEDERAL (SHCP)
# ==============================================================================
#
# OBJETIVO:
# Filtrar la serie XET30 ('Saldo de la Deuda Bruta del Sector Público Federal
# en Pesos'), expresada en miles de pesos nominales, manteniendo una sola unidad
# de medida y frecuencia mensual.
#
# CONTRATO DE SALIDA INDIVIDUAL:
# - Archivo: data/processed/deuda_bruta_mensual.parquet
# - Columnas: CICLO (YYYY-MM), DEUDA (float64, miles de pesos nominales)
# ------------------------------------------------------------------------------
MESES_SHCP = {
    "Enero": 1, "Febrero": 2, "Marzo": 3, "Abril": 4,
    "Mayo": 5, "Junio": 6, "Julio": 7, "Agosto": 8,
    "Septiembre": 9, "Octubre": 10, "Noviembre": 11, "Diciembre": 12
}

df_deuda_raw = pd.read_csv(RUTA_DEUDA, encoding="utf-8")
print("Registros crudos Deuda:", len(df_deuda_raw), "| Series distintas:", df_deuda_raw["CLAVE_DE_CONCEPTO"].nunique())

df_deuda_xet30 = df_deuda_raw[df_deuda_raw["CLAVE_DE_CONCEPTO"] == "XET30"].copy()

# Verificación de unidad de medida y tipo de información (Saldo en Miles de Pesos)
unidad_deuda = df_deuda_xet30["UNIDAD_DE_MEDIDA"].unique().tolist()
print("Serie XET30 -> Unidad:", unidad_deuda, "| Tipo:", df_deuda_xet30["TIPO_DE_INFORMACION"].unique().tolist())
assert unidad_deuda == ["Miles de Pesos"], f"Unidad inesperada en XET30: {unidad_deuda}"

mes_num_deuda = df_deuda_xet30["MES"].astype(str).str.strip().map(MESES_SHCP)
assert mes_num_deuda.notna().all(), "Existen nombres de mes no reconocidos en deuda_publica.csv"

df_deuda_xet30["CICLO"] = (
    df_deuda_xet30["CICLO"].astype(int).astype(str)
    + "-"
    + mes_num_deuda.astype(int).astype(str).str.zfill(2)
)
df_deuda_xet30["DEUDA"] = pd.to_numeric(df_deuda_xet30["MONTO"], errors="coerce")

deuda_parquet = (
    df_deuda_xet30[["CICLO", "DEUDA"]]
    .dropna(subset=["CICLO", "DEUDA"])
    .sort_values("CICLO", ascending=True)
    .drop_duplicates(subset=["CICLO"], keep="last")
    .reset_index(drop=True)
)

assert deuda_parquet["CICLO"].duplicated().sum() == 0

SALIDA_DEUDA = PROCESSED_DIR / "deuda_bruta_mensual.parquet"
deuda_parquet.to_parquet(SALIDA_DEUDA, index=False)

print(f"Deuda Bruta -> Filas: {len(deuda_parquet)} | Cobertura: {deuda_parquet['CICLO'].min()} a {deuda_parquet['CICLO'].max()}")
display(deuda_parquet.head(5))
display(deuda_parquet.tail(3))

Registros crudos Deuda: 84816 | Series distintas: 519
Serie XET30 -> Unidad: ['Miles de Pesos'] | Tipo: ['Saldo']
Deuda Bruta -> Filas: 186 | Cobertura: 2011-01 a 2026-06


,CICLO,DEUDA
0,2011-01,"4,430,293,041.0000"
1,2011-02,"4,503,709,153.0000"
2,2011-03,"4,534,934,949.0000"
3,2011-04,"4,546,027,059.0000"
4,2011-05,"4,585,947,148.0000"


,CICLO,DEUDA
183,2026-04,"20,015,928,407.0000"
184,2026-05,"20,156,542,927.0000"
185,2026-06,"20,354,849,720.0000"


In [26]:
# ==============================================================================
# 5. FUENTE 4 — INDICADOR GLOBAL DE LA ACTIVIDAD ECONÓMICA (IGAE - INEGI)
# ==============================================================================
#
# OBJETIVO:
# Extraer exclusivamente la serie 'Total' del bloque 'Índice de volumen físico,
# base 2018=100' desde la hoja 'Tabulado' del Excel de INEGI (IGAE_2.csv.xlsx),
# transformando el formato ancho (años en fila 4, meses en fila 5) a formato largo.
#
# CONTRATO DE SALIDA INDIVIDUAL:
# - Archivo: data/processed/igae_total_mensual.parquet
# - Columnas: CICLO (YYYY-MM), MONTO_IGAE_TOTAL (float64, índice base 2018=100)
# ------------------------------------------------------------------------------
raw_igae = pd.read_excel(RUTA_IGAE, sheet_name="Tabulado", header=None)
assert "Índice de volumen físico" in str(raw_igae.iloc[6, 0]), "No se localizó el bloque de índices en fila 6"

# Bloque de índices de volumen físico (filas 7 a 30)
filas_indices = raw_igae.iloc[7:31, :].copy()
filas_indices[0] = filas_indices[0].astype(str).str.strip()

fila_total = filas_indices[filas_indices[0] == "Total"]
assert len(fila_total) == 1, "No se encontró una fila única para 'Total' en IGAE"
fila_total = fila_total.iloc[0]

# Reconstruir eje temporal (años en fila 4 con forward-fill, meses en fila 5 ignorando 'Anual' y marcas P/R)
MESES_LOWER = {k.lower(): v for k, v in MESES_SHCP.items()}
anios_igae = raw_igae.iloc[4, :].ffill()

registros_igae = []
for col in range(1, raw_igae.shape[1]):
    texto_anio = str(anios_igae.iloc[col])
    texto_mes = str(raw_igae.iloc[5, col]).strip().lower()

    match_anio = re.search(r"(?:19|20)\d{2}", texto_anio)
    mes_num = next((num for nombre, num in MESES_LOWER.items() if texto_mes.startswith(nombre)), None)

    if match_anio and mes_num:
        valor = pd.to_numeric(fila_total[col], errors="coerce")
        if pd.notna(valor):
            ciclo_str = f"{int(match_anio.group()):04d}-{mes_num:02d}"
            registros_igae.append({"CICLO": ciclo_str, "MONTO_IGAE_TOTAL": float(valor)})

igae_parquet = (
    pd.DataFrame(registros_igae)
    .sort_values("CICLO", ascending=True)
    .drop_duplicates(subset=["CICLO"], keep="last")
    .reset_index(drop=True)
)

assert igae_parquet["CICLO"].duplicated().sum() == 0
assert (igae_parquet["MONTO_IGAE_TOTAL"] > 0).all()

SALIDA_IGAE = PROCESSED_DIR / "igae_total_mensual.parquet"
igae_parquet.to_parquet(SALIDA_IGAE, index=False)

print(f"IGAE Total -> Filas: {len(igae_parquet)} | Cobertura: {igae_parquet['CICLO'].min()} a {igae_parquet['CICLO'].max()}")
display(igae_parquet.head(5))
display(igae_parquet.tail(3))

IGAE Total -> Filas: 403 | Cobertura: 1993-01 a 2026-07


,CICLO,MONTO_IGAE_TOTAL
0,1993-01,55.4347
1,1993-02,56.4570
2,1993-03,58.9005
3,1993-04,57.1358
4,1993-05,57.8919


,CICLO,MONTO_IGAE_TOTAL
400,2026-05,108.5492
401,2026-06,107.3908
402,2026-07,109.2101


In [27]:
# ==============================================================================
# 6. FUENTE 5 — INGRESOS Y GASTO TOTAL DEL GOBIERNO FEDERAL (SHCP)
# ==============================================================================
#
# OBJETIVO:
# Leer 'ingreso_gasto_finan.csv' (~250 MB, +860 mil registros), seleccionar las
# columnas relevantes y filtrar la columna CLAVE_DE_CONCEPTO para conservar
# exclusivamente:
#   - XBB -> Ingresos del Gobierno Federal ('INGRESO PUBLICO')
#   - XBC -> Gasto Total del Gobierno Federal ('GASTO PUBLICO')
# Verificar que ambos estén en 'Miles de Pesos' nominales (misma unidad que DEUDA)
# y pivotar a formato mensual con una fila por CICLO.
#
# CONTRATO DE SALIDA INDIVIDUAL:
# - Archivo: data/processed/ingreso_gasto_mensual.parquet
# - Columnas: CICLO (YYYY-MM), INGRESO PUBLICO (float64), GASTO PUBLICO (float64)
# ------------------------------------------------------------------------------
columnas_ig = ["CICLO", "MES", "CLAVE_DE_CONCEPTO", "UNIDAD_DE_MEDIDA", "MONTO"]

try:
    df_ig_raw = pd.read_csv(RUTA_INGRESO_GASTO, usecols=columnas_ig, encoding="utf-8", low_memory=False)
except UnicodeDecodeError:
    df_ig_raw = pd.read_csv(RUTA_INGRESO_GASTO, usecols=columnas_ig, encoding="latin-1", low_memory=False)

print("Archivo leído:", RUTA_INGRESO_GASTO.name)
print("Filas totales en CSV crudo:", f"{len(df_ig_raw):,}")
print("Claves distintas en CSV crudo:", f"{df_ig_raw['CLAVE_DE_CONCEPTO'].nunique():,}")

# Normalizar espacios y filtrar exclusivamente las claves XBB (Ingreso) y XBC (Gasto)
df_ig_raw["CLAVE_DE_CONCEPTO"] = df_ig_raw["CLAVE_DE_CONCEPTO"].astype(str).str.strip()
df_ig_filtrado = df_ig_raw[df_ig_raw["CLAVE_DE_CONCEPTO"].isin(["XBB", "XBC"])].copy()

# Verificar unidad de medida homogénea con DEUDA ('Miles de Pesos')
unidades_ig = df_ig_filtrado["UNIDAD_DE_MEDIDA"].astype(str).str.strip().unique().tolist()
print("Claves conservadas:", df_ig_filtrado["CLAVE_DE_CONCEPTO"].unique().tolist(), "| Unidad:", unidades_ig)
assert unidades_ig == ["Miles de Pesos"], f"Unidad inesperada en Ingreso/Gasto: {unidades_ig}"

# Eliminar duplicados exactos antes de pivotar (como en Laboratorio 5)
df_ig_filtrado = df_ig_filtrado.drop_duplicates(subset=["CICLO", "MES", "CLAVE_DE_CONCEPTO"])

mes_num_ig = df_ig_filtrado["MES"].astype(str).str.strip().map(MESES_SHCP)
assert mes_num_ig.notna().all(), "Existen nombres de mes no reconocidos en ingreso_gasto_finan.csv"

df_ig_filtrado["CICLO"] = (
    df_ig_filtrado["CICLO"].astype(int).astype(str)
    + "-"
    + mes_num_ig.astype(int).astype(str).str.zfill(2)
)
df_ig_filtrado["MONTO"] = pd.to_numeric(df_ig_filtrado["MONTO"], errors="coerce")

# Pivotar para obtener una sola fila por mes con columnas INGRESO PUBLICO y GASTO PUBLICO
ingreso_gasto_parquet = (
    df_ig_filtrado
    .pivot(index="CICLO", columns="CLAVE_DE_CONCEPTO", values="MONTO")
    .reset_index()
    .rename(columns={
        "XBB": "INGRESO PUBLICO",
        "XBC": "GASTO PUBLICO"
    })
)
ingreso_gasto_parquet.columns.name = None
ingreso_gasto_parquet = (
    ingreso_gasto_parquet[["CICLO", "INGRESO PUBLICO", "GASTO PUBLICO"]]
    .sort_values("CICLO", ascending=True)
    .reset_index(drop=True)
)

assert ingreso_gasto_parquet["CICLO"].duplicated().sum() == 0

SALIDA_INGRESO_GASTO = PROCESSED_DIR / "ingreso_gasto_mensual.parquet"
ingreso_gasto_parquet.to_parquet(SALIDA_INGRESO_GASTO, index=False)

print(f"Ingreso/Gasto limpio -> Filas: {len(ingreso_gasto_parquet)} | Cobertura: {ingreso_gasto_parquet['CICLO'].min()} a {ingreso_gasto_parquet['CICLO'].max()}")
display(ingreso_gasto_parquet.head(5))
display(ingreso_gasto_parquet.tail(3))

Archivo leído: ingreso_gasto_finan.csv
Filas totales en CSV crudo: 863,376
Claves distintas en CSV crudo: 2,712
Claves conservadas: ['XBB', 'XBC'] | Unidad: ['Miles de Pesos']
Ingreso/Gasto limpio -> Filas: 187 | Cobertura: 2011-01 a 2026-07


,CICLO,INGRESO PUBLICO,GASTO PUBLICO
0,2011-01,"230,578,834.9000","224,760,906.3000"
1,2011-02,"174,860,300.0000","190,150,740.0000"
2,2011-03,"180,949,799.8000","198,574,645.6000"
3,2011-04,"188,091,731.0000","192,524,448.6000"
4,2011-05,"179,043,470.2000","193,785,245.8000"


,CICLO,INGRESO PUBLICO,GASTO PUBLICO
184,2026-05,"456,293,574.6000","685,471,135.8000"
185,2026-06,"542,570,508.5000","730,534,668.2000"
186,2026-07,"504,791,111.7000","698,363,305.7000"


In [28]:
# ==============================================================================
# 7. CUESTIONAMIENTO PEDAGÓGICO — UNIDADES, SALDOS VS. FLUJOS Y COBERTURA
# ==============================================================================
#
# PREGUNTA 1:
# ¿Por qué verificamos mediante assert que DEUDA, INGRESO PUBLICO y GASTO PUBLICO
# estén todos en 'Miles de Pesos', y por qué NO multiplicamos ni dividimos
# MONTO_IGAE_TOTAL, TIIE ni VALOR_INPC por 1,000?
#
# RESPUESTA EXPLICATIVA:
# - Homogeneidad de escala monetaria: En las bases de SHCP, tanto la deuda
#   bruta (XET30) como los ingresos (XBB) y gastos (XBC) ya vienen reportados
#   en 'Miles de Pesos' nominales. Si una serie viniera en pesos unitarios y
#   otra en miles o millones, cualquier razón financiera (como Deuda/Ingreso o
#   Déficit/Deuda) estaría distorsionada por factores de 1,000 o 1,000,000.
# - Naturaleza distinta de cada variable:
#   * DEUDA: Variable de ACERVO o SALDO (miles de pesos al cierre de cada mes).
#   * INGRESO PUBLICO y GASTO PUBLICO: Variables de FLUJO (miles de pesos
#     recaudados o ejercidos durante el mes).
#   * MONTO_IGAE_TOTAL y VALOR_INPC: NÚMEROS ÍNDICE adimensionales (base=100).
#   * TIIE: TASA DE INTERÉS porcentual anualizada (%).
#
# PREGUNTA 2:
# Si definimos el rango de análisis desde 2011-01 hasta 2026-07, ¿por qué el
# inner join de las 5 fuentes llega hasta 2026-06 (186 meses)?
#
# RESPUESTA EXPLICATIVA:
# Cada institución publica con distinto rezago oficial:
# - Deuda Pública (XET30) llega hasta 2026-06 (186 meses desde 2011-01).
# - Ingreso/Gasto e IGAE llegan hasta 2026-07 (187 meses).
# - INPC llega hasta 2026-08 y TIIE hasta 2026-10.
# Al exigir un INNER JOIN para garantizar que las 5 columnas tengan información
# completa sin valores nulos (NaN), la intersección temporal queda acotada por
# la serie con publicación más reciente más conservadora (2026-06).
# ==============================================================================

In [29]:
# ==============================================================================
# 8. INTEGRACIÓN FINAL — CONSOLIDACIÓN EN UN SOLO PARQUET (2011 EN ADELANTE)
# ==============================================================================
#
# OBJETIVO:
# Unir las 5 series mensuales mediante INNER JOIN sobre la clave común CICLO,
# respetando el orden exacto de columnas solicitado:
#   1. CICLO
#   2. DEUDA
#   3. INGRESO PUBLICO
#   4. GASTO PUBLICO
#   5. MONTO_IGAE_TOTAL
#   6. TIIE
#   7. VALOR_INPC
#
# CONTRATO DE SALIDA CONSOLIDADO:
# - Archivo: data/processed/macro_finanzas_consolidado.parquet
# - Rango: 2011-01 en adelante hasta donde coinciden todas las fuentes completas.
# ------------------------------------------------------------------------------
consolidado = (
    deuda_parquet
    .merge(ingreso_gasto_parquet, on="CICLO", how="inner")
    .merge(igae_parquet, on="CICLO", how="inner")
    .merge(tiie_parquet, on="CICLO", how="inner")
    .merge(inpc_parquet, on="CICLO", how="inner")
)

# Filtrar explícitamente de 2011-01 hasta 2026-07 y ordenar cronológicamente
consolidado = (
    consolidado[(consolidado["CICLO"] >= "2011-01") & (consolidado["CICLO"] <= "2026-07")]
    [["CICLO", "DEUDA", "INGRESO PUBLICO", "GASTO PUBLICO", "MONTO_IGAE_TOTAL", "TIIE", "VALOR_INPC"]]
    .sort_values("CICLO", ascending=True)
    .reset_index(drop=True)
)

# Validaciones estrictas de calidad antes de persistir
assert list(consolidado.columns) == [
    "CICLO", "DEUDA", "INGRESO PUBLICO", "GASTO PUBLICO", "MONTO_IGAE_TOTAL", "TIIE", "VALOR_INPC"
]
assert consolidado["CICLO"].duplicated().sum() == 0, "Existen ciclos duplicados en el consolidado"
assert consolidado.isna().sum().sum() == 0, "Existen valores nulos en el consolidado"

# Verificar continuidad mensual sin huecos entre el primer y último ciclo
ciclos_esperados = pd.period_range(
    start=consolidado["CICLO"].min(),
    end=consolidado["CICLO"].max(),
    freq="M"
).astype(str).tolist()
assert consolidado["CICLO"].tolist() == ciclos_esperados, "Faltan meses intermedios en la serie consolidada"

SALIDA_CONSOLIDADO = PROCESSED_DIR / "macro_finanzas_consolidado.parquet"
consolidado.to_parquet(SALIDA_CONSOLIDADO, index=False)

print("Consolidado guardado exitosamente en:", SALIDA_CONSOLIDADO)
print(f"Dimensiones: {consolidado.shape[0]} meses x {consolidado.shape[1]} columnas")
print(f"Periodo completo sin huecos: {consolidado['CICLO'].min()} -> {consolidado['CICLO'].max()}")
display(consolidado.head(6))
display(consolidado.tail(6))

Consolidado guardado exitosamente en: C:\Users\micha\Documents\VS Code\PyEconGDL\data\processed\macro_finanzas_consolidado.parquet
Dimensiones: 186 meses x 7 columnas
Periodo completo sin huecos: 2011-01 -> 2026-06


,CICLO,DEUDA,INGRESO PUBLICO,GASTO PUBLICO,MONTO_IGAE_TOTAL,TIIE,VALOR_INPC
0,2011-01,"4,430,293,041.0000","230,578,834.9000","224,760,906.3000",81.0595,4.5495,75.2960
1,2011-02,"4,503,709,153.0000","174,860,300.0000","190,150,740.0000",80.1156,4.5184,75.5785
2,2011-03,"4,534,934,949.0000","180,949,799.8000","198,574,645.6000",87.1737,4.4886,75.7235
3,2011-04,"4,546,027,059.0000","188,091,731.0000","192,524,448.6000",82.9572,4.5047,75.7174
4,2011-05,"4,585,947,148.0000","179,043,470.2000","193,785,245.8000",86.0386,4.4664,75.1593
5,2011-06,"4,611,783,138.0000","171,030,414.4000","286,911,217.0000",86.8752,4.4927,75.1555


,CICLO,DEUDA,INGRESO PUBLICO,GASTO PUBLICO,MONTO_IGAE_TOTAL,TIIE,VALOR_INPC
180,2026-01,"19,941,059,984.0000","651,619,119.6000","788,368,965.7000",100.9668,7.0338,143.5880
181,2026-02,"19,960,195,937.0000","469,697,985.1000","555,099,040.7000",100.1428,7.0274,144.3070
182,2026-03,"20,114,345,218.0000","639,765,699.5000","765,637,997.0000",106.7387,6.9729,145.5440
183,2026-04,"20,015,928,407.0000","556,618,185.0000","606,119,996.2000",106.6137,6.7590,145.8310
184,2026-05,"20,156,542,927.0000","456,293,574.6000","685,471,135.8000",108.5492,6.5550,145.5270
185,2026-06,"20,354,849,720.0000","542,570,508.5000","730,534,668.2000",107.3908,6.4777,145.1310


In [30]:
# ==============================================================================
# 9. AUDITORÍA FINAL DE ARCHIVOS PARQUET GENERADOS EN data/processed/
# ==============================================================================
#
# Comprobamos leyendo directamente desde disco que los 6 archivos Parquet
# (5 individuales con su cobertura propia + 1 consolidado) existen, no están
# vacíos y conservan los esquemas esperados.
# ------------------------------------------------------------------------------
archivos_salida = [
    ("INPC individual", SALIDA_INPC),
    ("TIIE individual", SALIDA_TIIE),
    ("Deuda Bruta individual", SALIDA_DEUDA),
    ("IGAE Total individual", SALIDA_IGAE),
    ("Ingreso y Gasto individual", SALIDA_INGRESO_GASTO),
    ("CONSOLIDADO FINAL", SALIDA_CONSOLIDADO),
]

resumen_auditoria = []
for nombre_serie, ruta_pq in archivos_salida:
    df_verif = pd.read_parquet(ruta_pq)
    resumen_auditoria.append({
        "Dataset": nombre_serie,
        "Archivo": ruta_pq.name,
        "Filas": len(df_verif),
        "Columnas": ", ".join(df_verif.columns),
        "Desde": df_verif["CICLO"].min(),
        "Hasta": df_verif["CICLO"].max(),
        "Nulos": int(df_verif.isna().sum().sum()),
        "Tamaño (KB)": round(ruta_pq.stat().st_size / 1024, 2),
    })

df_auditoria = pd.DataFrame(resumen_auditoria)
display(df_auditoria)

,Dataset,Archivo,Filas,Columnas,Desde,Hasta,Nulos,Tamaño (KB)
0,INPC individual,inpc_mensual.parquet,284,"CICLO, VALOR_INPC",2003-01,2026-08,0,5.5800
1,TIIE individual,tiie_mensual.parquet,190,"CICLO, TIIE",2011-01,2026-10,0,4.2200
2,Deuda Bruta individual,deuda_bruta_mensual.parquet,186,"CICLO, DEUDA",2011-01,2026-06,0,4.0300
3,IGAE Total individual,igae_total_mensual.parquet,403,"CICLO, MONTO_IGAE_TOTAL",1993-01,2026-07,0,7.2500
4,Ingreso y Gasto individual,ingreso_gasto_mensual.parquet,187,"CICLO, INGRESO PUBLICO, GASTO PUBLICO",2011-01,2026-07,0,6.1200
5,CONSOLIDADO FINAL,macro_finanzas_consolidado.parquet,186,"CICLO, DEUDA, INGRESO PUBLICO, GASTO PUBLICO, ...",2011-01,2026-06,0,14.7600


In [ ]:
# ==============================================================================
# RESUMEN DE ARQUITECTURA Y SIGUIENTE ETAPA
# ==============================================================================
#
#    LO QUE CONSTRUIMOS EN ESTA ETAPA:
#    - Desacoplamos la ingesta específica de 5 fuentes institucionales (INEGI,
#      Banxico y SHCP), respetando la codificación y estructura real de cada una.
#    - Generamos 5 productos individuales en formato Parquet dentro de
#      PyEconGDL/data/processed/, conservando el horizonte temporal propio de
#      cada serie y el orden cronológico ascendente por CICLO (YYYY-MM).
#    - Integramos las 5 fuentes en un dataset consolidado limpio y sin nulos
#      (macro_finanzas_consolidado.parquet), listo para la siguiente etapa de
#      enriquecimiento y deflactación con VALOR_INPC.
#
# ==============================================================================